# Weeks 3–4 · Method B – Character reading
Run on Colab with a GPU (Runtime → Change runtime type → T4).
Training outputs are saved directly to Google Drive so nothing is lost if the session drops.

## 0. Setup: mount Drive, clone the repo, install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/DATH_bien_so'
BRANCH = 'nhom2/ocr-training'

!git clone -b $BRANCH https://github.com/anhpham103/Programming-Integration-Project_nhan_dang_bien_so.git /content/repo || git -C /content/repo pull
%cd /content/repo
!pip -q install ultralytics onnx onnxslim onnxruntime

## 1. Copy the character dataset from Drive (week 2 output)

In [ ]:
!mkdir -p datasets
!rsync -a --info=progress2 $DRIVE/datasets/chars/ datasets/chars/
!cp $DRIVE/datasets/plate_strings.csv datasets/
!ls datasets/chars/images/*/ | head; python -m pytest tests/ -q

## Week 3
### 2. Enhance (CLAHE) all images, check labels, character statistics

In [ ]:
!python scripts/enhance_chars_dataset.py

### 3. Train YOLO26n, 30 classes, imgsz 320, fliplr=0 (about 2–4 hours on a T4)

In [ ]:
!python scripts/train_chars.py --project $DRIVE/runs/detect --device 0

### 3b. Only if the session dropped: resume from last.pt

In [ ]:
!python scripts/train_chars.py --project $DRIVE/runs/detect --resume

### 4. Try read_chars on a few validation images (handoff H4)

In [ ]:
import sys, cv2, glob
sys.path[:0] = ['src', 'scripts']
from compare_ab import load_strings
from read_chars import read_chars, set_model
set_model('models/char_yolo26n.pt')
gts = load_strings('datasets/plate_strings.csv')
for p in sorted(glob.glob('datasets/chars_enh/images/val/*'))[:10]:
    stem = p.rsplit('/',1)[1].rsplit('.',1)[0]
    gt = gts.get(stem, '?')
    print(f"{stem:<30} read={read_chars(cv2.imread(p), '|' in gt):<12} gt={gt}")

## Week 4
### 5. Compare A and B on the validation split (needs Bao's rectify.py, postprocess.py, read_baseline)

In [ ]:
!pip -q install paddlepaddle paddleocr
!python scripts/compare_ab.py --split val

### 6. Export ONNX and check it reads the same as .pt (handoff H6)

In [ ]:
!python scripts/export_chars_onnx.py
!python scripts/compare_ab.py --split val --engines B --model models/char_yolo26n.onnx --tag onnx

### 7. Prepare 32x160 line images for method C

In [ ]:
!python scripts/prepare_lines_c.py
import cv2, glob, matplotlib.pyplot as plt
fs = sorted(glob.glob('datasets/lines/val/*_l1.png'))[:4] + sorted(glob.glob('datasets/lines/val/*_l0.png'))[:4]
plt.figure(figsize=(8, 4))
for i, f in enumerate(fs):
    plt.subplot(4, 2, i + 1); plt.imshow(cv2.imread(f)[:, :, ::-1]); plt.title(f.rsplit('/',1)[1], fontsize=7); plt.axis('off')
plt.tight_layout()

### 8. Save models, results and line images to Drive

In [ ]:
!mkdir -p $DRIVE/models $DRIVE/results $DRIVE/datasets
!cp models/char_yolo26n.* $DRIVE/models/
!cp results/*.csv $DRIVE/results/
!rsync -a datasets/lines/ $DRIVE/datasets/lines/